# Finnhub connectivity test

Smoke test for the Finnhub `company-news` endpoint (the backtesting sentiment source
 described in `02_research/alternative_data_sources.md`).

**Setup:** add your free Finnhub key (from [finnhub.io](https://finnhub.io)) to either

- environment variable `FINNHUB_API_KEY`, or
- a new line in `config/credentials.env` formatted as `FINNHUB_API_KEY=your_key_here`

Then run all cells. The test fetches recent headlines for a single ticker, normalizes
them to a long-format dataframe, and asserts the response is well-formed. No look-ahead
concerns here — this only checks live connectivity and schema; time alignment happens
later in the feature store via `merge_asof(..., direction='backward')`.

In [3]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd
import finnhub

In [4]:
api_key = os.getenv("FINNHUB_API_KEY")

if not api_key:
    root = Path.cwd().resolve()
    while root != root.parent and not (root / "config" / "credentials.env").exists():
        root = root.parent

    credentials_path = root / "config" / "credentials.env"
    if credentials_path.is_file():
        for ln in credentials_path.read_text(encoding="utf-8").splitlines():
            ln = ln.strip()
            if ln.startswith("FINNHUB_API_KEY="):
                api_key = ln.split("=", 1)[1].strip()
                break

if not api_key:
    raise RuntimeError(
        "Set FINNHUB_API_KEY env var or add FINNHUB_API_KEY=... to config/credentials.env"
    )

client = finnhub.Client(api_key=api_key)
print("finnhub client ready (key ends with ...%s)" % api_key[-4:])

finnhub client ready (key ends with ...fln0)


In [5]:
# --- Fetch recent company news for a single ticker ---
TICKER = "AAPL"
END = pd.Timestamp.utcnow().normalize()
START = END - pd.Timedelta(days=7)

raw = client.company_news(
    TICKER,
    _from=START.strftime("%Y-%m-%d"),
    to=END.strftime("%Y-%m-%d"),
)

print(f"fetched {len(raw)} articles for {TICKER} "
      f"({START.date()} -> {END.date()})")
if raw:
    print("sample keys:", sorted(raw[0].keys()))

fetched 247 articles for AAPL (2026-06-25 -> 2026-07-02)
sample keys: ['category', 'datetime', 'headline', 'id', 'image', 'related', 'source', 'summary', 'url']


In [6]:
# --- Normalize to a long-format dataframe (one row per article) ---
EXPECTED_FIELDS = ("datetime", "headline", "summary", "source", "url", "id")


def news_to_long(articles: list[dict], ticker: str) -> pd.DataFrame:
    if not articles:
        return pd.DataFrame(columns=["ticker", "published_at", *EXPECTED_FIELDS])
    df = pd.DataFrame(articles)
    df["ticker"] = ticker
    # Finnhub 'datetime' is a UNIX epoch (seconds, UTC)
    df["published_at"] = pd.to_datetime(df["datetime"], unit="s", utc=True)
    cols = ["ticker", "published_at", *[c for c in EXPECTED_FIELDS if c in df.columns]]
    return df[cols].sort_values("published_at").reset_index(drop=True)


news = news_to_long(raw, TICKER)
news.head()

,ticker,published_at,datetime,headline,summary,source,url,id
0,AAPL,2026-06-27 19:32:17+00:00,1782588737,Apple (AAPL) Plans Mac Chip Roadmap Shift Towa...,Apple Inc. (NASDAQ:AAPL) is one of the 15 Best...,Yahoo,https://finnhub.io/api/news?id=feb629ed2d86e17...,140748106
1,AAPL,2026-06-28 02:53:28+00:00,1782615208,Markets Starting To Choke On Massive Surge In ...,Massive equity and AI-related debt issuanceâ...,SeekingAlpha,https://finnhub.io/api/news?id=5f66f86ae8e56ee...,140752419
2,AAPL,2026-06-28 06:49:07+00:00,1782629347,"Amkor Technology, Inc. (AMKR) Is A “Secret Wea...","Amkor Technology, Inc. (NASDAQ:AMKR) is one of...",Yahoo,https://finnhub.io/api/news?id=52839bf28bdfe28...,140752723
3,AAPL,2026-06-28 06:49:24+00:00,1782629364,Berkshire May Just Save You From A Likely Mark...,Berkshire Hathaway is well-positioned to outpe...,SeekingAlpha,https://finnhub.io/api/news?id=065427260e90b56...,140753383
4,AAPL,2026-06-28 09:00:00+00:00,1782637200,Even Apple supply chain maestro Tim Cook could...,"Apple, Microsoft, HP and other gadget makers a...",Yahoo,https://finnhub.io/api/news?id=2e1eacbc58d06e7...,140754111


In [7]:
# --- Assertions: connectivity + schema + basic quality ---
assert isinstance(raw, list), f"expected list response, got {type(raw)}"
assert len(raw) > 0, (
    "Finnhub returned zero articles. Either the key is invalid/rate-limited, "
    "or the date window has no news — widen START/END and retry."
)

for field in ("datetime", "headline"):
    assert field in news.columns or field in raw[0], f"missing field '{field}' in response"

assert news["headline"].notna().all(), "some articles have a null headline"
assert news["published_at"].notna().all(), "some articles have a null timestamp"
assert (news["published_at"] <= pd.Timestamp.utcnow() + pd.Timedelta(days=1)).all(), (
    "found article timestamps in the future"
)

print("PASS — Finnhub connectivity and schema OK")
print(f"  ticker={TICKER}  articles={len(news)}  "
      f"span={news['published_at'].min()} -> {news['published_at'].max()}")

PASS — Finnhub connectivity and schema OK
  ticker=AAPL  articles=247  span=2026-06-27 19:32:17+00:00 -> 2026-07-02 13:53:39+00:00


## Next steps

Once this passes, the deterministic path from `alternative_data_sources.md` is:
score each headline locally with VADER (`vaderSentiment`) or FinBERT, aggregate to a
daily per-ticker score, cache to parquet under `01_data/cache/`, and merge into the feature
store with `merge_asof(..., direction='backward')` so no future sentiment leaks into past
bars.